# Chapter 5 — Robot Motion: Velocity & Odometry Motion Models

**Why banana distribution?**
Noise is injected into *arc parameters* (v, ω) then pushed through a *nonlinear* polar→Cartesian
transform. The forward direction accumulates positional error, the angular component shears the cloud.

**This notebook covers:**
1. Velocity model — banana distribution, per-αᵢ sweep
2. Odometry model — side-by-side comparison
3. Edge cases: ω→0, v→0, large rotation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import matplotlib.pyplot as plt
from pluto_filters.pluto_filters.motion_models.velocity_motion_model import (
    sample_motion_model_velocity, motion_model_velocity
)

def sample_velocity(pose, v, omega, dt, alphas, N=1000):
    """Draw N samples from velocity motion model."""
    a1, a2, a3, a4, a5, a6 = alphas
    xs, ys = [], []
    for _ in range(N):
        v_hat  = v     + np.random.normal(0, np.sqrt(a1*v**2 + a2*omega**2))
        w_hat  = omega + np.random.normal(0, np.sqrt(a3*v**2 + a4*omega**2))
        g_hat  =         np.random.normal(0, np.sqrt(a5*v**2 + a6*omega**2))
        if abs(w_hat) < 1e-6:
            x2 = pose[0] + v_hat * dt * np.cos(pose[2])
            y2 = pose[1] + v_hat * dt * np.sin(pose[2])
        else:
            r  = v_hat / w_hat
            x2 = pose[0] - r*np.sin(pose[2]) + r*np.sin(pose[2] + w_hat*dt)
            y2 = pose[1] + r*np.cos(pose[2]) - r*np.cos(pose[2] + w_hat*dt)
        xs.append(x2); ys.append(y2)
    return np.array(xs), np.array(ys)

START = np.array([0.0, 0.0, 0.0])
V, OMEGA, DT = 0.5, 0.2, 3.0
ALPHAS_DEFAULT = (0.1, 0.01, 0.01, 0.1, 0.0, 0.0)
print("Imports OK")

In [ ]:
# ── Banana distributions: low / default / high noise ────────────────────────
configs = [
    ('Low noise',     (0.01, 0.001, 0.001, 0.01, 0, 0), 'green'),
    ('Default noise', (0.10, 0.010, 0.010, 0.10, 0, 0), 'steelblue'),
    ('High noise',    (0.40, 0.050, 0.050, 0.40, 0, 0), 'crimson'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Banana Distribution — Velocity Motion Model', fontsize=13)

for ax, (label, alphas, color) in zip(axes, configs):
    xs, ys = sample_velocity(START, V, OMEGA, DT, alphas, N=2000)
    ax.scatter(xs, ys, s=3, alpha=0.4, color=color)
    ax.plot(*START[:2], 'k*', ms=12, label='Start')
    # Expected final pose (noiseless)
    if abs(OMEGA) > 1e-6:
        r = V / OMEGA
        xf = START[0] - r*np.sin(START[2]) + r*np.sin(START[2]+OMEGA*DT)
        yf = START[1] + r*np.cos(START[2]) - r*np.cos(START[2]+OMEGA*DT)
    ax.plot(xf, yf, 'k+', ms=12, mew=3, label='Expected end')
    ax.set_title(f'{label}'); ax.legend(fontsize=8); ax.set_aspect('equal')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout(); plt.savefig('ch05_banana.png', dpi=120); plt.show()

## Per-αᵢ sweep — which parameter controls which shape?

In [ ]:
# Per-αᵢ sweep: vary one αᵢ at a time, hold others at default
BASE = [0.10, 0.01, 0.01, 0.10, 0.0, 0.0]
SCALE = 5.0  # high value to make effect visible

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Per-αᵢ Sweep — What Does Each Parameter Control?', fontsize=13)
descriptions = [
    'α₁: translational noise from v',
    'α₂: translational noise from ω',
    'α₃: rotational noise from v',
    'α₄: rotational noise from ω (main)',
    'α₅: final heading bias from v',
    'α₆: final heading bias from ω',
]

for i, (ax, desc) in enumerate(zip(axes.flat, descriptions)):
    alphas_hi = BASE.copy(); alphas_hi[i] *= SCALE
    alphas_lo = BASE.copy(); alphas_lo[i] /= SCALE
    xs_hi, ys_hi = sample_velocity(START, V, OMEGA, DT, tuple(alphas_hi), N=1500)
    xs_lo, ys_lo = sample_velocity(START, V, OMEGA, DT, tuple(alphas_lo), N=1500)
    ax.scatter(xs_lo, ys_lo, s=3, alpha=0.4, color='green',   label=f'αᵢ×0.2')
    ax.scatter(xs_hi, ys_hi, s=3, alpha=0.4, color='crimson', label=f'αᵢ×{SCALE}')
    ax.plot(*START[:2], 'k*', ms=10)
    ax.set_title(desc, fontsize=9); ax.legend(fontsize=7); ax.set_aspect('equal')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout(); plt.savefig('ch05_alpha_sweep.png', dpi=120); plt.show()

## Odometry Model — Side-by-Side with Velocity Model

In [ ]:
# Odometry motion model (Table 5.6 from Thrun et al.)
def sample_odometry(pose, odom_delta, alphas, N=2000):
    """
    odom_delta = (delta_rot1, delta_trans, delta_rot2)
    Noise injected into each odometry component.
    """
    a1, a2, a3, a4 = alphas
    drot1, dtrans, drot2 = odom_delta
    xs, ys = [], []
    for _ in range(N):
        dr1 = drot1  - np.random.normal(0, np.sqrt(a1*drot1**2  + a2*dtrans**2))
        dt  = dtrans - np.random.normal(0, np.sqrt(a3*dtrans**2 + a4*(drot1**2+drot2**2)))
        dr2 = drot2  - np.random.normal(0, np.sqrt(a1*drot2**2  + a2*dtrans**2))
        x2  = pose[0] + dt * np.cos(pose[2] + dr1)
        y2  = pose[1] + dt * np.sin(pose[2] + dr1)
        xs.append(x2); ys.append(y2)
    return np.array(xs), np.array(ys)

# Equivalent motion: 1.5m forward with slight arc
ODOM_DELTA  = (0.1, 1.5, 0.1)   # rot1=0.1rad, trans=1.5m, rot2=0.1rad
ODOM_ALPHAS = (0.1, 0.01, 0.01, 0.1)

xs_vel, ys_vel = sample_velocity(START, 0.5, 0.05, 3.0, ALPHAS_DEFAULT, N=2000)
xs_odom, ys_odom = sample_odometry(START, ODOM_DELTA, ODOM_ALPHAS, N=2000)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Velocity Model vs Odometry Model — Noise Shape Comparison', fontsize=13)

for ax, xs, ys, title, color in [
    (axes[0], xs_vel, ys_vel, 'Velocity model\nnoise on (v, ω)', 'steelblue'),
    (axes[1], xs_odom, ys_odom, 'Odometry model\nnoise on (Δrot₁, Δtrans, Δrot₂)', 'darkorange'),
]:
    ax.scatter(xs, ys, s=4, alpha=0.4, color=color)
    ax.plot(*START[:2], 'k*', ms=12, label='Start')
    ax.set_title(title); ax.legend(fontsize=8); ax.set_aspect('equal')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_xlim(-1, 3); ax.set_ylim(-1.5, 1.5)

plt.tight_layout(); plt.savefig('ch05_vel_vs_odom.png', dpi=120); plt.show()
print("Key difference:")
print("Velocity model: noise on continuous (v,ω) → smooth banana")
print("Odometry model: noise on discrete (rot1,trans,rot2) → similar shape, different parameterization")

## Edge Cases: ω→0, v→0, large rotation

In [ ]:
# EDGE CASES
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Edge Cases — Motion Model Boundary Conditions', fontsize=13)

# Case 1: ω→0 (near-straight line)
omegas = [1e-6, 0.01, 0.2]
ax = axes[0]
colors_ = ['green', 'steelblue', 'crimson']
for omega_, col in zip(omegas, colors_):
    xs_, ys_ = sample_velocity(START, 0.5, omega_, 3.0, ALPHAS_DEFAULT, N=500)
    ax.scatter(xs_, ys_, s=4, alpha=0.5, color=col, label=f'ω={omega_}')
ax.plot(*START[:2], 'k*', ms=12); ax.set_title('ω→0: straight line (no banana shape)')
ax.legend(fontsize=8); ax.set_aspect('equal'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

# Case 2: v→0 (rotation in place)
vs = [0.001, 0.05, 0.3]
ax = axes[1]
for v_, col in zip(vs, colors_):
    xs_, ys_ = sample_velocity(START, v_, 1.0, 3.0, ALPHAS_DEFAULT, N=500)
    ax.scatter(xs_, ys_, s=4, alpha=0.5, color=col, label=f'v={v_}')
ax.plot(*START[:2], 'k*', ms=12); ax.set_title('v→0: rotation in place')
ax.legend(fontsize=8); ax.set_aspect('equal'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

# Case 3: large rotation (ωdt = π)
ax = axes[2]
for omega_, col in zip([np.pi/3, np.pi/2, np.pi], colors_):
    xs_, ys_ = sample_velocity(START, 0.5, omega_, 1.0, ALPHAS_DEFAULT, N=500)
    ax.scatter(xs_, ys_, s=4, alpha=0.5, color=col, label=f'ω={omega_/np.pi:.1f}π rad/s')
ax.plot(*START[:2], 'k*', ms=12); ax.set_title('Large rotation: arc wraps around')
ax.legend(fontsize=8); ax.set_aspect('equal'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout(); plt.savefig('ch05_edge_cases.png', dpi=120); plt.show()

## ✅ Compare | 🔥 Break | 📏 Measure

In [ ]:
# COMPARE: velocity vs odometry model spread
xs_v, ys_v   = sample_velocity(START, 0.5, 0.1, 3.0, ALPHAS_DEFAULT, N=5000)
xs_o, ys_o   = sample_odometry(START, ODOM_DELTA, ODOM_ALPHAS, N=5000)
print("=== COMPARE: spread of final positions ===")
for label, xs_, ys_ in [('Velocity model', xs_v, ys_v), ('Odometry model', xs_o, ys_o)]:
    print(f"  {label}: σ_x={xs_.std():.4f}m  σ_y={ys_.std():.4f}m")

# BREAK: what happens with NEGATIVE velocity?
print("\n=== BREAK: negative velocity (reverse motion) ===")
xs_rev, ys_rev = sample_velocity(START, -0.5, 0.2, 3.0, ALPHAS_DEFAULT, N=2000)
print(f"  Forward (v=+0.5): mean_x={xs_v.mean():.3f}m")
print(f"  Reverse (v=-0.5): mean_x={xs_rev.mean():.3f}m  (should be negative)")

# MEASURE: probability density at expected final pose
if abs(OMEGA) > 1e-6:
    r_ = V / OMEGA
    xf_ = START[0] - r_*np.sin(START[2]) + r_*np.sin(START[2]+OMEGA*DT)
    yf_ = START[1] + r_*np.cos(START[2]) - r_*np.cos(START[2]+OMEGA*DT)
    tf_ = START[2] + OMEGA * DT

print("\n=== MEASURE: empirical statistics ===")
for label, alphas in [('Low', (0.01,0.001,0.001,0.01,0,0)),
                       ('Default', ALPHAS_DEFAULT),
                       ('High', (0.4,0.05,0.05,0.4,0,0))]:
    xs_, ys_ = sample_velocity(START, V, OMEGA, DT, alphas, N=5000)
    cov = np.cov(np.vstack([xs_, ys_]))
    print(f"  {label:10s}: σ_x={xs_.std():.4f}m  σ_y={ys_.std():.4f}m  det(cov)={np.linalg.det(cov):.6f}")